In [2]:

!pip install qiskit qiskit-aer

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
import matplotlib.pyplot as plt
import random

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 116.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 90.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 4.3 MB/s eta 0:00:00


**Creates a 1-qubit circuit, applies Hadamard gate to put it in superposition, then measures it. Returns 0 or 1**

In [3]:
def get_quantum_random_bit():

    # Create a circuit with 1 qubit and 1 classical bit
    qc = QuantumCircuit(1, 1)

    # Apply Hadamard gate → qubit enters superposition (50/50 state)
    qc.h(0)

    # Measure the qubit → collapses to 0 or 1 randomly
    qc.measure(0, 0)

    # Use Aer simulator (mimics a real quantum computer)
    simulator = AerSimulator()
    job = simulator.run(qc, shots=1)
    result = job.result()

    # Extract the measured bit
    counts = result.get_counts(qc)
    bit = int(list(counts.keys())[0])
    return bit

# Test it
print("Random quantum bit:", get_quantum_random_bit())

Random quantum bit: 1


**Creates an n-qubit circuit. Each qubit goes through a Hadamard gate and is measured independently.**

In [4]:
def get_quantum_random_bits(n_bits=8):

    qc = QuantumCircuit(n_bits, n_bits)

    # Apply Hadamard to ALL qubits at once
    for i in range(n_bits):
        qc.h(i)

    # Measure all qubits
    qc.measure(range(n_bits), range(n_bits))

    simulator = AerSimulator()
    job = simulator.run(qc, shots=1)
    result = job.result()

    counts = result.get_counts(qc)
    bitstring = list(counts.keys())[0]
    return bitstring

# Test it
bits = get_quantum_random_bits(8)
print(f"8-bit quantum random string: {bits}")
print(f"As integer (0-255): {int(bits, 2)}")

8-bit quantum random string: 01010000
As integer (0-255): 80


**Generates a cryptographic key using quantum randomness.**


*  key_length_bytes=16 → 128-bit key (AES-128 standard)
*  key_length_bytes=32 → 256-bit key (AES-256 standard)



In [5]:
def generate_quantum_key(key_length_bytes=16):

    key_bits = key_length_bytes * 8
    bits = get_quantum_random_bits(key_bits)

    # Convert bitstring to hexadecimal (standard key format)
    key_int = int(bits, 2)
    hex_key = format(key_int, f'0{key_length_bytes*2}x')
    return hex_key

# Generate a 128-bit key
key = generate_quantum_key(16)
print(f"Quantum-generated 128-bit encryption key:")
print(f"  {key}")
print(f"  Length: {len(key)*4} bits")

Quantum-generated 128-bit encryption key:
  6db06bf1e95511cfc843801539752159
  Length: 128 bits


**Draws the quantum circuit**

In [6]:

def visualize_circuit(n_bits=4):

    qc = QuantumCircuit(n_bits, n_bits)
    for i in range(n_bits):
        qc.h(i)
    qc.measure(range(n_bits), range(n_bits))

    print("Quantum Circuit Diagram:")
    print(qc.draw(output='text'))

visualize_circuit(4)

Quantum Circuit Diagram:
     ┌───┐┌─┐         
q_0: ┤ H ├┤M├─────────
     ├───┤└╥┘┌─┐      
q_1: ┤ H ├─╫─┤M├──────
     ├───┤ ║ └╥┘┌─┐   
q_2: ┤ H ├─╫──╫─┤M├───
     ├───┤ ║  ║ └╥┘┌─┐
q_3: ┤ H ├─╫──╫──╫─┤M├
     └───┘ ║  ║  ║ └╥┘
c: 4/══════╩══╩══╩══╩═
           0  1  2  3 


**Encrypts a message using a quantum-generated One-Time Pad.This is mathematically unbreakable if key is truly random.**

In [7]:
def quantum_otp_encrypt(message: str) -> tuple:

    # Convert message to bytes
    msg_bytes = message.encode('utf-8')
    n_bits_needed = len(msg_bytes) * 8

    # Generate quantum key (same length as message)
    key_bits = get_quantum_random_bits(n_bits_needed)
    key_int = int(key_bits, 2)

    # Convert key to bytes
    key_bytes = key_int.to_bytes(len(msg_bytes), byteorder='big')

    # XOR message with key → ciphertext
    ciphertext = bytes([m ^ k for m, k in zip(msg_bytes, key_bytes)])

    print(f"Original message : {message}")
    print(f"Quantum key (hex): {key_bytes.hex()}")
    print(f"Ciphertext (hex) : {ciphertext.hex()}")
    return ciphertext, key_bytes

def quantum_otp_decrypt(ciphertext: bytes, key_bytes: bytes) -> str:
    """Decrypts using the same key — XOR again."""
    decrypted = bytes([c ^ k for c, k in zip(ciphertext, key_bytes)])
    return decrypted.decode('utf-8')

# Test it
cipher, key = quantum_otp_encrypt("HELLO QUANTUM")
recovered = quantum_otp_decrypt(cipher, key)
print(f"Decrypted back   : {recovered}")

Original message : HELLO QUANTUM
Quantum key (hex): 0e72612518dfd46927debdf7ef
Ciphertext (hex) : 46372d6957ff853c6690e9a2a2
Decrypted back   : HELLO QUANTUM


**For Mersenne Twister and Quantum RNG**

* watches 624 outputs from the generator

* Feeds all 624 numbers into RandCrack tool
*  RandCrack solves algebra: "what 624 internal numbers produced these outputs?"


In [21]:
!pip install randcrack -q

import random
from randcrack import RandCrack
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

# ── Quantum generator ──────────────────────────────────────────
def get_qrng_32bit():
    qc = QuantumCircuit(32, 32)
    for i in range(32):
        qc.h(i)
    qc.measure(range(32), range(32))
    result = AerSimulator().run(qc, shots=1).result()
    bitstring = list(result.get_counts(qc).keys())[0]
    return int(bitstring[::-1], 2)

# ── RandCrack attack on MT ─────────────────────────────────────
def attack_mersenne_twister():
    print("═"*55)
    print("  ATTACK 1: Mersenne Twister (Python random)")
    print("═"*55)
    print("  Step 1: Attacker observes 624 outputs...")

    rc = RandCrack()
    observed = []
    for _ in range(624):
        # CRITICAL: must use getrandbits(32) — RandCrack expects this exactly
        val = random.getrandbits(32)
        observed.append(val)
        rc.submit(val)

    print("  Step 2: Predicting next 100 outputs...")
    correct = 0
    for _ in range(100):
        predicted = rc.predict_getrandbits(32)
        actual    = random.getrandbits(32)
        if predicted == actual:
            correct += 1

    acc = correct / 100 * 100
    print(f"  Correct: {correct}/100")
    print(f"  Accuracy: {acc:.0f}%")
    print(f"  Verdict: {'FULLY CRACKED ← state cloned perfectly' if correct > 90 else 'partial'}")


# ── Bit-level prediction on QRNG ──────────────────────────────
def attack_quantum():
    print("\n" + "═"*55)
    print("  ATTACK 2: Quantum RNG (Qiskit)")
    print("═"*55)
    print("  Step 1: Attacker observes 624 quantum outputs...")

    observed = []
    for i in range(624):
        observed.append(get_qrng_32bit())
        if (i+1) % 100 == 0:
            print(f"    collected {i+1}/624...")

    # Try to find ANY pattern in observed values
    # Simplest attack: does next value correlate with previous?
    print("  Step 2: Trying to predict next 32 outputs...")
    print("  (using last observed value as guess — simplest possible attack)")
    correct = 0
    for i in range(32):
        # Attacker's best guess: use the last seen value
        # (if quantum had memory, this would work)
        guess  = observed[-1]
        actual = get_qrng_32bit()
        if guess == actual:
            correct += 1

    acc = correct / 32 * 100
    print(f"  Correct: {correct}/32")
    print(f"  Accuracy: {acc:.1f}%")

    print(f"  Verdict: {'has memory — not random' if correct > 2 else 'UNCRACKABLE — no memory, no pattern'}")


# ── Run both ───────────────────────────────────────────────────
attack_mersenne_twister()
attack_quantum()



═══════════════════════════════════════════════════════
  ATTACK 1: Mersenne Twister (Python random)
═══════════════════════════════════════════════════════
  Step 1: Attacker observes 624 outputs...
  Step 2: Predicting next 100 outputs...
  Correct: 100/100
  Accuracy: 100%
  Verdict: FULLY CRACKED ← state cloned perfectly

═══════════════════════════════════════════════════════
  ATTACK 2: Quantum RNG (Qiskit)
═══════════════════════════════════════════════════════
  Step 1: Attacker observes 624 quantum outputs...
    collected 100/624...
    collected 200/624...
    collected 300/624...
    collected 400/624...
    collected 500/624...
    collected 600/624...
  Step 2: Trying to predict next 32 outputs...
  (using last observed value as guess — simplest possible attack)
  Correct: 0/32
  Accuracy: 0.0%
  Verdict: UNCRACKABLE — no memory, no pattern


No training involved anywhere.
RandCrack does not learn or train. It solves a system of equations.
Mersenne Twister's output → equations have a solution → state recovered.
Quantum output → equations have no solution → nothing recovered.